# Quantum Phase Estimation: 2-Qubit Hamiltonian Eigenvalue Estimation
**Course module: AyushDocs-HamiltonianEstimation**

In the previous notebook, we estimated the phase of a single-qubit unitary gate. In this notebook, we scale up the QPE algorithm to estimate the eigenvalues of a physical quantum system — a **2-qubit Hamiltonian**.

---

## 1. Physical Model: The 2-Qubit Hamiltonian

Consider the following diagonal Hamiltonian representing two interacting spins in a magnetic field:

$$H = 0.5 (Z \otimes Z) + 0.3 (Z \otimes I)$$

In Qiskit, qubits are ordered in a little-endian convention. The state $|q_1 q_0\rangle$ is indexed such that $q_0$ is the rightmost qubit (least significant bit). Let's calculate the analytical eigenvalues for the four computational basis states (which are the eigenstates of this diagonal Hamiltonian):

1. **State $|00\rangle$** ($q_0 = 0, q_1 = 0$):
   $$H |00\rangle = \left(0.5 (Z|0\rangle \otimes Z|0\rangle) + 0.3 (Z|0\rangle \otimes I|0\rangle)\right) = (0.5 + 0.3) |00\rangle = 0.8 |00\rangle$$
   *Eigenvalue $\lambda = 0.8$
2. **State $|10\rangle$** ($q_0 = 1, q_1 = 0$):
   $$H |10\rangle = \left(0.5 (Z|0\rangle \otimes Z|1\rangle) + 0.3 (Z|1\rangle \otimes I|0\rangle)\right) = (-0.5 - 0.3) |10\rangle = -0.8 |10\rangle$$
   *Eigenvalue $\lambda = -0.8$
3. **State $|01\rangle$** ($q_0 = 0, q_1 = 1$):
   $$H |01\rangle = \left(0.5 (Z|1\rangle \otimes Z|0\rangle) + 0.3 (Z|0\rangle \otimes I|1\rangle)\right) = (-0.5 + 0.3) |01\rangle = -0.2 |01\rangle$$
   *Eigenvalue $\lambda = -0.2$
4. **State $|11\rangle$** ($q_0 = 1, q_1 = 1$):
   $$H |11\rangle = \left(0.5 (Z|1\rangle \otimes Z|1\rangle) + 0.3 (Z|1\rangle \otimes I|1\rangle)\right) = (0.5 - 0.3) |11\rangle = 0.2 |11\rangle$$
   *Eigenvalue $\lambda = 0.2$

These values form our true physical eigenvalues:
$$\text{HAM\_EIGVALS} = [0.8, -0.8, -0.2, 0.2]$$

---

## 2. Generalizing QPE to Hamiltonian Simulation

To estimate the eigenvalues of $H$, we choose the unitary operator $U$ to be the time-evolution operator:

$$U = e^{-i H t}$$

If $|\psi_k\rangle$ is an eigenstate of $H$ with eigenvalue $\lambda_k$, then it is also an eigenstate of $U$:

$$U |\psi_k\rangle = e^{-i \lambda_k t} |\psi_k\rangle = e^{2\pi i \phi_k} |\psi_k\rangle$$

Comparing the exponents, we have:

$$\phi_k \equiv -\frac{\lambda_k t}{2\pi} \pmod 1$$

Once QPE estimates the phase $\phi_k$, we can reconstruct the physical eigenvalue $\lambda_k$ via:

$$\lambda_k \approx -\frac{2\pi (\phi_k + m)}{t}$$

where $m \in \mathbb{Z}$ is selected to pull $\lambda_k$ into the valid energy range.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, DiagonalGate
from qiskit_aer import AerSimulator

## 3. Implementing the QPE Circuit for the Hamiltonian

Since our Hamiltonian $H$ is diagonal in the computational basis, the time evolution operator $e^{-i H t \cdot 2^j}$ is also diagonal. We can implement it using Qiskit's `DiagonalGate` controlled by the corresponding ancilla qubit.

In [2]:
HAM_EIGVALS = np.array([0.8, -0.8, -0.2, 0.2])

def time_evolve(t, power=1):
    """
    Computes the diagonal entries of the time evolution operator exp(-i * H * t * power).
    """
    return np.exp(-1j * HAM_EIGVALS * t * power)

def qpe_hamiltonian(t, num_ancilla, eigenstate=0):
    """
    Constructs the QPE circuit to estimate the eigenvalue of a chosen eigenstate (0 to 3).
    """
    n = num_ancilla
    qc = QuantumCircuit(n + 2, n)  # n ancillas + 2 target qubits
    target = [n, n + 1]
    ancillas = list(range(n))

    # Prepare target qubits in the chosen eigenstate
    for i in range(2):
        if (eigenstate >> i) & 1:
            qc.x(target[i])

    # Superposition on ancillas
    qc.h(ancillas)

    # Apply controlled-DiagonalGate for each power of U
    for j in range(n):
        diag = time_evolve(t, 2**j).tolist()
        gate = DiagonalGate(diag)
        cgate = gate.control(1)
        qc.append(cgate, [ancillas[j]] + target)

    # Inverse QFT
    qc.compose(QFTGate(n).inverse(), ancillas, inplace=True)

    # Measurement
    qc.measure(ancillas, list(range(n)))
    return qc

## 4. Reconstructing Physical Eigenvalues

In [3]:
SIM = AerSimulator()

def _run(qc, shots):
    qct = transpile(qc, SIM)
    return SIM.run(qct, shots=shots).result().get_counts()

def run_qpe_ham(t, num_ancilla, eigenstate=0, shots=20000):
    qc = qpe_hamiltonian(t, num_ancilla, eigenstate)
    counts = _run(qc, shots)
    total = sum(counts.values())

    # Weighted average phase estimate
    phase_est = 0.0
    for bits, c in counts.items():
        frac = sum(int(bits[i]) / (1 << (i + 1)) for i in range(len(bits)))
        phase_est += frac * c / total

    true_val = HAM_EIGVALS[eigenstate]
    true_phase = (-true_val * t / (2 * np.pi)) % 1.0

    # Eigenvalue reconstruction: λ = -2π(φ + k)/t
    # Shift the estimation to physical window [-0.8, 0.8]
    period = 2 * np.pi / t
    eig_est = -2 * np.pi * phase_est / t
    while eig_est < min(HAM_EIGVALS) - 0.1:
        eig_est += period
    while eig_est > max(HAM_EIGVALS) + 0.1:
        eig_est -= period

    eig_err = abs(eig_est - true_val)
    return eig_est, true_val, eig_err, counts

## 5. Simulating All Eigenstates

Let's choose $t = \pi/2$ and use $n=8$ ancilla qubits. We will run QPE on each of the four possible input eigenstates $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ and print the results in a formatted table.

In [4]:
t = np.pi / 2
n_ham = 8
SHOTS = 20000

print(f"Executing QPE Hamiltonian Eigenvalue Estimation (t = {t:.4f}, n = {n_ham})")
print(f"{'State':>6} | {'True λ':>8} | {'Estimated λ':>12} | {'Error':>10}")
print("-" * 46)
for state in range(4):
    eig_est, eig_true, eig_err, _ = run_qpe_ham(t, n_ham, state, SHOTS)
    print(f"  |{state:02b}⟩   | {eig_true:+8.2f} | {eig_est:+12.6f} | {eig_err:10.2e}")

Executing QPE Hamiltonian Eigenvalue Estimation (t = 1.5708, n = 8)
 State |   True λ |  Estimated λ |      Error
----------------------------------------------


  |00⟩   |    +0.80 |    +0.799969 |   3.13e-05


  |01⟩   |    -0.80 |    -0.799799 |   2.01e-04


  |10⟩   |    -0.20 |    -0.211673 |   1.17e-02


  |11⟩   |    +0.20 |    +0.214180 |   1.42e-02


### Discussion
We successfully reconstructed the physical eigenvalues of the Hamiltonian using the phase values obtained from QPE. This shows the general applicability of QPE to system identification and energy estimation tasks, which is the cornerstone of VQE validation and quantum chemistry simulations.